In [3]:
import os
import truststore

os.environ["AWS_PROFILE"] = "NP-DP-SysAccounts-AllPlatforms"
os.environ["AWS_REGION"] = "ap-southeast-2"
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"

truststore.inject_into_ssl()

from anthropic import AnthropicBedrock

client = AnthropicBedrock(
    aws_region="ap-southeast-2"
)

model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"

In [10]:
def add_user_message(messages,text):
    user_message = {'role':'user','content':text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {'role':'assistant','content':text}
    messages.append(assistant_message)

def chat(messages,system=None,stop_sequences=None):
#def chat(messages):
    params = {
        "model": model,
        "max_tokens":1000,
        "messages":messages}
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    
    return message.content[0].text 

Without Prefilling + Stop Sequence 

In [8]:
messages = []
add_user_message(messages, 'Generate a very short event bridge in JSON format')
#add_assistant_message(messages, "")

answer = chat(messages)
print(answer)

```json
{
  "version": "0",
  "id": "12345678-1234-1234-1234-123456789012",
  "detail-type": "EC2 Instance State-change Notification",
  "source": "aws.ec2",
  "account": "123456789012",
  "time": "2024-01-15T10:30:00Z",
  "region": "us-east-1",
  "detail": {
    "instance-id": "i-1234567890abcdef0",
    "state": "running"
  }
}
```


Prefilling + STOP sequence

In [11]:
messages = []
add_user_message(messages, 'Generate a very short event bridge in JSON format')
add_assistant_message(messages, "```json")

answer = chat(messages, stop_sequences=["```"])
print(answer)


{
  "DetailType": "User Action",
  "Source": "myapp",
  "Detail": {
    "action": "login",
    "userId": "user123",
    "timestamp": "2024-01-15T10:30:00Z"
  }
}



In [13]:
messages = []
add_user_message(messages, 'Generate a very short csv table with fact table student with 3 rows')
add_assistant_message(messages, "```csv")

answer = chat(messages,stop_sequences=["```"]) 
print(answer)


student_id,name,grade,age
1,Alice Johnson,A,18
2,Bob Smith,B,17
3,Carol White,A,18

